# Unit 4 exercises: co-occurrence, item-item cosine, and k-NN collaborative filtering

Run the recsys data generators before you begin. These exercises rebuild the first *collaborative* path from the lesson -- a binary reader x item incidence of the train positives, an item-item cosine co-occurrence similarity, then k-nearest-neighbour retrieval over it -- reveal the `bookrec` implementation, and score it honestly on the Unit 1 scoreboard against the random floor, popularity, and the Unit-3 lexical path. Collaborative filtering reads only *who read what*: it never opens the catalog text. A reader's `seen` set is always their positive `train` history; `val` is the sealed holdout we grade against. Each answer is left for you to complete; no solutions live here.

In [ ]:
import json

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

# The train positives are both the co-occurrence signal and each reader's `seen` set.
train = interactions[interactions["split"] == "train"]
train_positive = train[train["label"] == 1]
seen_by = (
    train_positive.groupby("reader_id")["item_id"]
    .apply(lambda items: {int(i) for i in items})
    .to_dict()
)

# Map each catalog book to its column in the reader x item matrices below.
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}

## Exercise 1 - Implicit feedback by the numbers: positives, exposure-sampled negatives, and what co-occurrence reads

The interaction log is *implicit* feedback: a reader either engaged a book or did not, and a `label == 1` row is a clear positive while a `label == 0` row is an **exposure-sampled negative** (the reader was shown the book and did not engage it) standing in for the unobserved "did not like". There are no explicit negative *ratings*. Count the train split three ways: its total rows, its `label == 1` positives (readers engaged the book), and its `label == 0` sampled negatives, and record what fraction of the train log is positive. Then record, from those counts alone, how many train rows feed the item-item co-occurrence similarity and how many it ignores -- the co-occurrence path is built from **positives only** (`split == "train"` and `label == 1`) and ranks the full catalog, sampling no negatives.

This exercise is reasoning and counting over the log, not path behaviour. In your explanation (two or three sentences) say why only the positives enter co-occurrence (a `label == 0` row carries no co-*liking* signal), confirm that Unit 4 does **no** negative sampling, and name the later units that actually *use* the `label == 0` sampled negatives as training signal.

In [ ]:
n_train = None            # TODO: number of rows in the train split
n_positive = None         # TODO: train rows with label == 1 (readers engaged the book)
n_negative = None         # TODO: train rows with label == 0 (exposed, not engaged)
positive_share = None     # TODO: n_positive / n_train

# These describe the SAME train log; the co-occurrence similarity reads only one group.
rows_entering_cooccurrence = None   # TODO: how many train rows feed the item-item similarity
rows_ignored_by_cooccurrence = None # TODO: how many train rows the similarity ignores

positives_only_explanation = "TODO"  # 2-3 sentences: why only label==1 positives enter co-occurrence;
                                      # that Unit 4 does NO negative sampling (full-catalog ranking);
                                      # which later units USE the label==0 sampled negatives for training

## Exercise 2 - Build the binary reader x item incidence of the train positives

Collaborative filtering starts from a binary **reader x item incidence** matrix $\mathbf{R}$ built from the train positives, where $\mathbf{R}_{r,i} = 1$ exactly when reader $r$ has a train positive for item $i$ and $0$ otherwise. Collect the set of `(reader_id, item_id)` positive pairs (as ints), index the readers that appear in the positives in **sorted** order (readers are the rows), and take the **whole catalog** as the item universe (columns, via `col_of`). Fill a `float` incidence array of shape `(n_readers, n_items)` with a $1.0$ for each pair.

Record the matrix shape, the number of non-zero entries (which must equal the number of distinct positive pairs), and -- as a plain-English read-back -- the column sum for book $307$, i.e. how many readers liked it. Verify that `int(incidence.sum())` equals the number of pairs.

In [ ]:
pairs = None              # TODO: set of (int(reader_id), int(item_id)) over the train positives
reader_ids = None         # TODO: sorted list of the distinct reader ids appearing in `pairs`
reader_of = None          # TODO: {reader_id: row index} in that sorted order

incidence = None          # TODO: np.zeros((len(reader_ids), len(catalog_ids)));
                          #       set incidence[reader_of[r], col_of[i]] = 1.0 for each (r, i) in pairs
incidence_shape = None    # TODO: incidence.shape  (readers, items)
n_pairs = None            # TODO: int(incidence.sum())
readers_of_book_307 = None  # TODO: int(incidence[:, col_of[307]].sum()) -- how many readers liked book 307
sum_matches_pairs = None  # TODO: bool - int(incidence.sum()) == len(pairs)

## Exercise 3 - The item-item cosine co-occurrence similarity, by hand, then reveal `bookrec.item_item_cosine`

Two books co-occur for every reader who liked both -- the dot product of their incidence columns, the plain count of shared readers. Raw co-occurrence favours popular books, so we normalize by each column's length, exactly the cosine between two *item* columns:

$$\mathrm{sim}(i, j) = \frac{\mathbf{R}_{:,i} \cdot \mathbf{R}_{:,j}}{\lVert \mathbf{R}_{:,i} \rVert \, \lVert \mathbf{R}_{:,j} \rVert}.$$

Compute the column norms $\lVert \mathbf{R}_{:,i} \rVert = \sqrt{\sum_r \mathbf{R}_{r,i}^2}$, form the item-item dot products $\mathbf{R}^{\top}\mathbf{R}$, and divide by the outer product of the norms clamped to a tiny floor (a book nobody read has norm $0$, which would otherwise divide by zero). Zero the **diagonal** so a book is never its own neighbour. Read one cell back as the count it encodes: the shared readers of books $307$ and $1066$ and their cosine. Verify the diagonal is all zero and that the matrix is symmetric (so $\mathrm{sim}(i,j) = \mathrm{sim}(j,i)$). Then reveal `bookrec.item_item_cosine(incidence)`, which ships exactly this computation, and confirm it agrees with your by-hand matrix to floating-point precision.

In [ ]:
norms = None              # TODO: np.sqrt((incidence ** 2).sum(axis=0)) -- ||column_i|| = sqrt(#readers of i)
sim_by_hand = None        # TODO: (incidence.T @ incidence) / np.maximum(np.outer(norms, norms), 1e-9)
# TODO: zero the diagonal of sim_by_hand in place with np.fill_diagonal(sim_by_hand, 0.0)

i, j = col_of[307], col_of[1066]
shared_readers = None     # TODO: int((incidence[:, i] * incidence[:, j]).sum()) -- co-readers of 307 and 1066
sim_307_1066 = None       # TODO: sim_by_hand[i, j]
diagonal_is_zero = None   # TODO: bool - np.allclose(np.diag(sim_by_hand), 0.0)
is_symmetric = None       # TODO: bool - np.allclose(sim_by_hand, sim_by_hand.T)

sim_lib = None            # TODO: bookrec.item_item_cosine(incidence)
matches_library = None    # TODO: bool - np.allclose(sim_lib, sim_by_hand)

## Exercise 4 - k-nearest-neighbour retrieval: score candidates by summed similarity to a reader's history

To recommend *for a reader*, use their history as the query: a candidate book scores by the **summed similarity** to every book the reader has already liked,

$$\mathrm{score}(c) = \sum_{s \,\in\, \mathrm{seen}} \mathrm{sim}(c, s).$$

This is k-nearest-neighbour retrieval over the item-similarity matrix. Take reader $3$, whose `seen` set is `seen_by[3]`. Gather the columns of the seen books and sum the similarity rows across them to score every catalog book. Wrap each book as a `bookrec.Candidate(item_id, float(score), "cooc-by-hand")` and call `bookrec.rank(candidates, n=5, exclude=seen)` -- `exclude=seen` drops the books the reader has already read, because a re-read is not a recommendation (the `n` here is how many recommendations to return, not the `n_neighbors` cap of the challenge exercises). Record the top-5 item ids and verify none of them is in `seen`. In one sentence, say why excluding the seen books is the right default.

In [ ]:
reader_id = 3
seen = None               # TODO: seen_by[reader_id] -- the reader's train-positive books
seen_cols = None          # TODO: [col_of[item_id] for item_id in sorted(seen)]
scores = None             # TODO: sim_by_hand[:, seen_cols].sum(axis=1) -- summed similarity to the history

candidates = None         # TODO: [bookrec.Candidate(item_id, float(scores[col]), "cooc-by-hand")
                          #        for col, item_id in enumerate(catalog_ids)]
top5 = None               # TODO: bookrec.rank(candidates, n=5, exclude=seen)
top5_ids = None           # TODO: [c.item_id for c in top5]
no_seen_recommended = None  # TODO: bool - none of top5_ids is in seen (what exclude=seen guarantees)
exclude_explanation = "TODO"  # one sentence: why a re-read is not a recommendation

## Exercise 5 - Reveal `ItemItemRetrievalPath`: reproduce the by-hand ranking, and the honest empty-history answer

The package packages the by-hand pieces as a standard retrieval path. `bookrec.ItemItemRetrievalPath()` fits the item-item cosine similarity from the **train positives** of the log -- `fit(interactions, catalog=None)` duck-types the row-mapping log exactly like `PopularityRetrievalPath` (no pandas inside the package) and honours `catalog=` as the item universe. `retrieve(query, context, k)` identifies the reader through `context["seen"]`, scores each candidate by the summed similarity to the seen books, excludes the seen books, and returns the top `k`. Fit it on `rows` with `catalog=catalog_ids`, retrieve reader $3$'s top-5, and confirm it reproduces the by-hand `top5_ids` from Exercise 4 (same ids, same order). Then retrieve for a reader with an **empty** `seen` set and confirm the answer is `[]` -- the honest answer from a path that depends entirely on history. In one sentence, say which earlier (Unit 2) path would step in for a reader with no history.

In [ ]:
cf = None                 # TODO: bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
shipped = None            # TODO: cf.retrieve(query=reader_id, context={"seen": seen}, k=5)
shipped_ids = None        # TODO: [c.item_id for c in shipped]
reproduces_by_hand = None # TODO: bool - shipped_ids == top5_ids (the by-hand ranking from Exercise 4)

empty_history = None      # TODO: cf.retrieve(query=999_999, context={"seen": set()}, k=5)
empty_is_empty = None     # TODO: bool - empty_history == []
cold_start_explanation = "TODO"  # one sentence: which Unit 2 path steps in for a reader with no history

## Exercise 6 - Register the paths and read the validation scoreboard honestly

Line CF up against the paths of the earlier units under the Unit 1 rules unchanged. Build a `bookrec.RandomRetrievalPath(catalog_ids, seed=0)` floor, fit a `bookrec.PopularityRetrievalPath()` and a `bookrec.LexicalRetrievalPath(keywords)` (load the keyword corpus first) on `rows` with `catalog=catalog_ids`, and register all four -- random, popularity, lexical, and the `cf` from Exercise 5 -- in a fresh `bookrec.PathRegistry`. Score each with `bookrec.run_validation_scoreboard(path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)` and record the `hit_rate_at_k` of each, plus CF's ratio over popularity (expect ~2.3), over lexical (expect ~1.6), and over the random floor (expect ~21).

Then write two or three honest sentences. CF is **not** the first path to beat popularity -- Unit 3's lexical path already modestly beat it (~1.46x), and the lesson said so. CF is the first **collaborative** path, and the first to beat **both** popularity and the content/lexical path, by the widest margin so far. Say what the ratios do and do not establish, and do not claim "first to beat popularity".

In [ ]:
keywords = None           # TODO: bookrec.load_keywords(data_dir / "keywords.csv.gz")
random_floor = None       # TODO: bookrec.RandomRetrievalPath(catalog_ids, seed=0)
popularity = None         # TODO: bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = None            # TODO: bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
registry = None           # TODO: bookrec.PathRegistry(); register random_floor, popularity, lexical, cf


def score(path):
    return None            # TODO: bookrec.run_validation_scoreboard(
                           #           path, interactions_path, catalog_ids=catalog_ids,
                           #           k=10, cold_readers=cold_readers)


cf_hit = None             # TODO: score(cf).hit_rate_at_k
popularity_hit = None     # TODO: score(popularity).hit_rate_at_k
lexical_hit = None         # TODO: score(lexical).hit_rate_at_k
floor_hit = None          # TODO: score(random_floor).hit_rate_at_k
cf_vs_popularity = None   # TODO: cf_hit / popularity_hit   (expect ~2.3)
cf_vs_lexical = None      # TODO: cf_hit / lexical_hit      (expect ~1.6)
cf_vs_floor = None        # TODO: cf_hit / floor_hit        (expect ~21)
scoreboard_reading = "TODO"  # 2-3 honest sentences: NOT "first to beat popularity" (lexical already does,
                             # ~1.46x); first COLLABORATIVE path and first to beat BOTH popularity and lexical

## Exercise 7 - Challenge: the neighbour cap *degrades* the score here

The name "k-nearest-neighbour" suggests keeping only each book's top few neighbours and zeroing the rest -- a smaller, cleaner neighbourhood, offered by `ItemItemRetrievalPath` as an optional `n_neighbors` cap. On this data it is worth *measuring* rather than assuming. Fit `bookrec.ItemItemRetrievalPath(n_neighbors=10)` on `rows` with `catalog=catalog_ids`, score it on `val` with the same `score` helper from Exercise 6, and compare its hit@10 against the uncapped `cf_hit`. **Expect the cap to lower the score** -- do not expect an improvement. Record both numbers and a bool that the capped score is below the uncapped one, and explain in one or two sentences why a tight neighbourhood throws away real co-occurrence signal on this sparse log, so the shipped default is uncapped.

In [ ]:
capped = None             # TODO: bookrec.ItemItemRetrievalPath(n_neighbors=10).fit(rows, catalog=catalog_ids)
capped_hit = None         # TODO: score(capped).hit_rate_at_k
uncapped_hit = None       # TODO: cf_hit from Exercise 6 (the uncapped default)
cap_degrades = None       # TODO: bool - capped_hit < uncapped_hit  (expect True: the cap HURTS here)
cap_explanation = "TODO"  # 1-2 sentences: why keeping only the top-10 neighbours discards real
                          # co-occurrence signal on this sparse log, so the default is uncapped

## Exercise 8 - Challenge: the coverage ceiling, and the low-rank bridge to matrix factorization

A collaborative path can only recommend a book that *someone* co-read: a book with no train positives is an all-zero column, gets an all-zero similarity row, and can never be a neighbour of anything. Using the library similarity `sim_lib` from Exercise 3, compute each book's total similarity (its row sum) and count how many books are **reachable** (row sum greater than $0$) versus **unreachable** (no co-readership). Record the reachable count, the unreachable count, and the unreachable fraction of the catalog (expect ~0.41).

Then, in two or three sentences, interpret the ceiling honestly using the relation the lesson taught. First, mirror it against the lexical path's strength (lexical can recommend a brand-new book from its words alone, with no readers), which is why blending the two signals is Unit 6's job. Second, explain the bridge to Unit 5 as a **rank** argument: the item-item similarity is the normalized **Gram matrix** $\mathbf{R}^{\top}\mathbf{R}$ of the reader x item incidence $\mathbf{R}$, so its rank is at most $\mathrm{rank}(\mathbf{R})$; **matrix factorization** approximates that structure with a *deliberately smaller* latent rank (the low-rank view) that generalizes co-occurrence for the **warm** books that already have interactions. Note the limit precisely: this low-rank view still cannot place the unreachable **cold** books (zero train positives), so do **not** claim matrix factorization closes the coverage ceiling -- that gap waits for Unit 6 blending and Unit 9 feature-based cold start.

In [ ]:
row_sums = None           # TODO: sim_lib.sum(axis=1) -- the total similarity each book carries
reachable = None          # TODO: int((row_sums > 0).sum()) -- books with some co-readership
unreachable = None        # TODO: len(catalog_ids) - reachable -- books with no train positives
unreachable_share = None  # TODO: unreachable / len(catalog_ids)  (expect ~0.41)
coverage_explanation = "TODO"  # 2-3 sentences: how the ~41% ceiling mirrors lexical's strength, why it argues
                               # for blending (Unit 6), and why co-occurrence is a memorized pairwise
                               # (rank-reduced) view of taste that Unit 5 matrix factorization generalizes